In [4]:
# -- Cora Dataset --
import time
import random
from torch_geometric.datasets import Planetoid
import torch_geometric.utils as utils
import networkx as nx
from sklearn.metrics import roc_auc_score, average_precision_score

dataset = Planetoid(root='./data/Cora', name="Cora")
data = dataset[0]
test_ratio = 0.1

# 전체 그래프 networkx graph
g_nx = utils.to_networkx(data, to_undirected=True)
all_edges = list(g_nx.edges())
num_test_edges = int(len(all_edges) * test_ratio) # 


# 테스트용 양성 샘플 선택 후 그래프에서 제거 (학습용 그래프 생성)
random.seed(42)
test_edges_pos = random.sample(all_edges, num_test_edges)
train_g = g_nx.copy()
train_g.remove_edges_from(test_edges_pos)

# 음성 샘플 샘플링(=pos)
test_edges_neg = []
nodes = list(g_nx.nodes())
while len(test_edges_neg) < num_test_edges:
    u, v = random.sample(nodes, 2)
    if u != v and not g_nx.has_edge(u, v):
        test_edges_neg.append((u, v))

def get_pagerank_scores(graph, edge_list):
    scores = []
    
    # 각 노드별 PPR 결과를 캐싱하여 반복 계산을 줄입니다.
    # 대규모 그래프에서는 모든 노드를 미리 계산하는 것이 메모리/시간 면에서 효율적일 수 있습니다.
    ppr_cache = {}
    
    for u, v in edge_list:
        # 노드 u에서 시작하는 Personalized PageRank 계산
        if u not in ppr_cache:
            # alpha는 덤핑 팩터(Damping Factor)로, 보통 0.85를 사용합니다.
            # 빌트인 nx.pagerank에서 personalization 딕셔너리를 주면 PPR이 됩니다.
            ppr_cache[u] = nx.pagerank(graph, alpha=0.85, personalization={u: 1})
        
        # u에서 출발해 v에 도달할 확률 점수 저장
        scores.append(ppr_cache[u].get(v, 0.0))
    
    return scores

start_time = time.time()
pos_scores = get_pagerank_scores(train_g, test_edges_pos)
neg_scores = get_pagerank_scores(train_g, test_edges_neg)

y_true = [1] * len(pos_scores) + [0] * len(neg_scores)
y_scores = pos_scores + neg_scores

auc = roc_auc_score(y_true, y_scores)
ap = average_precision_score(y_true, y_scores)
print(f'--test pos edge number: {num_test_edges}, test neg edge number: {len(test_edges_neg)}')
print(f"Planetoid Cora 데이터셋 기준 Pagerank 링크 예측 AUC: {auc:.4f}, AP: {ap:.4f}")
print(f"걸린 시간: {time.time() - start_time:.4f} (s)")



--test pos edge number: 527, test neg edge number: 527
Planetoid Cora 데이터셋 기준 Pagerank 링크 예측 AUC: 0.8560, AP: 0.9000
걸린 시간: 8.4718 (s)


In [5]:
# -- Citeseer Dataset --
import time
import random
from torch_geometric.datasets import Planetoid
import torch_geometric.utils as utils
import networkx as nx
from sklearn.metrics import roc_auc_score, average_precision_score

dataset = Planetoid(root='./data/Citeseer', name="Citeseer")
data = dataset[0]
test_ratio = 0.1

# 전체 그래프 networkx graph
g_nx = utils.to_networkx(data, to_undirected=True)
all_edges = list(g_nx.edges())
num_test_edges = int(len(all_edges) * test_ratio) # 


# 테스트용 양성 샘플 선택 후 그래프에서 제거 (학습용 그래프 생성)
random.seed(42)
test_edges_pos = random.sample(all_edges, num_test_edges)
train_g = g_nx.copy()
train_g.remove_edges_from(test_edges_pos)

# 음성 샘플 샘플링(=pos)
test_edges_neg = []
nodes = list(g_nx.nodes())
while len(test_edges_neg) < num_test_edges:
    u, v = random.sample(nodes, 2)
    if u != v and not g_nx.has_edge(u, v):
        test_edges_neg.append((u, v))

def get_pagerank_scores(graph, edge_list):
    scores = []
    
    # 각 노드별 PPR 결과를 캐싱하여 반복 계산을 줄입니다.
    # 대규모 그래프에서는 모든 노드를 미리 계산하는 것이 메모리/시간 면에서 효율적일 수 있습니다.
    ppr_cache = {}
    
    for u, v in edge_list:
        # 노드 u에서 시작하는 Personalized PageRank 계산
        if u not in ppr_cache:
            # alpha는 덤핑 팩터(Damping Factor)로, 보통 0.85를 사용합니다.
            # 빌트인 nx.pagerank에서 personalization 딕셔너리를 주면 PPR이 됩니다.
            ppr_cache[u] = nx.pagerank(graph, alpha=0.85, personalization={u: 1})
        
        # u에서 출발해 v에 도달할 확률 점수 저장
        scores.append(ppr_cache[u].get(v, 0.0))
    
    return scores

start_time = time.time()
pos_scores = get_pagerank_scores(train_g, test_edges_pos)
neg_scores = get_pagerank_scores(train_g, test_edges_neg)

y_true = [1] * len(pos_scores) + [0] * len(neg_scores)
y_scores = pos_scores + neg_scores

auc = roc_auc_score(y_true, y_scores)
ap = average_precision_score(y_true, y_scores)
print(f'--test pos edge number: {num_test_edges}, test neg edge number: {len(test_edges_neg)}')
print(f"Planetoid Citeseer 데이터셋 기준 Pagerank 링크 예측 AUC: {auc:.4f}, AP: {ap:.4f}")
print(f"걸린 시간: {time.time() - start_time:.4f} (s)")



--test pos edge number: 455, test neg edge number: 455
Planetoid Citeseer 데이터셋 기준 Pagerank 링크 예측 AUC: 0.7256, AP: 0.8315
걸린 시간: 7.6797 (s)


In [7]:
# -- Pubmed Dataset --
import time
import random
from torch_geometric.datasets import Planetoid
import torch_geometric.utils as utils
import networkx as nx
from sklearn.metrics import roc_auc_score, average_precision_score

dataset = Planetoid(root='./data/Pubmed', name="Pubmed")
data = dataset[0]
test_ratio = 0.1

# 전체 그래프 networkx graph
g_nx = utils.to_networkx(data, to_undirected=True)
all_edges = list(g_nx.edges())
num_test_edges = int(len(all_edges) * test_ratio) # 


# 테스트용 양성 샘플 선택 후 그래프에서 제거 (학습용 그래프 생성)
random.seed(42)
test_edges_pos = random.sample(all_edges, num_test_edges)
train_g = g_nx.copy()
train_g.remove_edges_from(test_edges_pos)

# 음성 샘플 샘플링(=pos)
test_edges_neg = []
nodes = list(g_nx.nodes())
while len(test_edges_neg) < num_test_edges:
    u, v = random.sample(nodes, 2)
    if u != v and not g_nx.has_edge(u, v):
        test_edges_neg.append((u, v))

def get_pagerank_scores(graph, edge_list):
    scores = []
    
    # 각 노드별 PPR 결과를 캐싱하여 반복 계산을 줄입니다.
    # 대규모 그래프에서는 모든 노드를 미리 계산하는 것이 메모리/시간 면에서 효율적일 수 있습니다.
    ppr_cache = {}
    
    for u, v in edge_list:
        # 노드 u에서 시작하는 Personalized PageRank 계산
        if u not in ppr_cache:
            # alpha는 덤핑 팩터(Damping Factor)로, 보통 0.85를 사용합니다.
            # 빌트인 nx.pagerank에서 personalization 딕셔너리를 주면 PPR이 됩니다.
            ppr_cache[u] = nx.pagerank(graph, alpha=0.85, personalization={u: 1})
        
        # u에서 출발해 v에 도달할 확률 점수 저장
        scores.append(ppr_cache[u].get(v, 0.0))
    
    return scores

start_time = time.time()
pos_scores = get_pagerank_scores(train_g, test_edges_pos)
neg_scores = get_pagerank_scores(train_g, test_edges_neg)

y_true = [1] * len(pos_scores) + [0] * len(neg_scores)
y_scores = pos_scores + neg_scores

auc = roc_auc_score(y_true, y_scores)
ap = average_precision_score(y_true, y_scores)
print(f'--test pos edge number: {num_test_edges}, test neg edge number: {len(test_edges_neg)}')
print(f"Planetoid Pubmed 데이터셋 기준 Pagerank 링크 예측 AUC: {auc:.4f}, AP: {ap:.4f}")
print(f"걸린 시간: {time.time() - start_time:.4f} (s)")



--test pos edge number: 4432, test neg edge number: 4432
Planetoid Pubmed 데이터셋 기준 Pagerank 링크 예측 AUC: 0.8314, AP: 0.8883
걸린 시간: 748.0960 (s)


In [ ]:
import random
import time
import networkx as nx
from sklearn.metrics import roc_auc_score, average_precision_score

# 1. Davis Southern Women (2-mode) 데이터셋 로드
G = nx.davis_southern_women_graph()

# Node Set U (여성)와 Node Set V (이벤트) 분리
women = [n for n, d in G.nodes(data=True) if d['bipartite'] == 0]
events = [n for n, d in G.nodes(data=True) if d['bipartite'] == 1]

# 전체 엣지 및 테스트 비율 설정
all_edges = list(G.edges())
test_ratio = 0.2  # 테스트 비율 20%
num_test_edges = int(len(all_edges) * test_ratio)

# 2. 테스트용 양성 샘플(Pos) 선택 및 학습용 그래프(Train Graph) 생성
random.seed(42)
test_edges_pos = random.sample(all_edges, num_test_edges)

train_g = G.copy()
train_g.remove_edges_from(test_edges_pos)

# 3. 음성 샘플(Neg) 샘플링 (2-mode 조건 준수: u in U, v in V 간 미연결 쌍만 추출)
test_edges_neg = []
while len(test_edges_neg) < num_test_edges:
    u = random.choice(women)
    v = random.choice(events)
    if not G.has_edge(u, v) and (u, v) not in test_edges_neg:
        test_edges_neg.append((u, v))


# 4. Bipartite Personalised PageRank 계산 함수 정의
def get_bipartite_pagerank_scores(graph, edge_list, alpha=0.85):
    """
    각 시작 노드(u)를 기준(Personalization)으로 PageRank를 수행하여
    목적지 노드(v)에 도달할 확률 점수를 산출
    """
    scores = []
    
    # 시작 노드별로 Personalized PageRank 계산 결과를 캐싱하여 연산 최적화
    ppr_cache = {}

    for u, v in edge_list:
        # 노드 타입 정렬 (u: women, v: events)
        if u in events and v in women:
            u, v = v, u

        # u 노드 기준 Personalized PageRank 계산 (최초 1회만 연산 후 캐싱)
        if u not in ppr_cache:
            # u 노드에서 출발할 확률을 1.0으로 설정하는 personalization 벡터
            personalization = {node: 0.0 for node in graph.nodes()}
            personalization[u] = 1.0
            
            try:
                ppr_cache[u] = nx.pagerank(graph, alpha=alpha, personalization=personalization)
            except nx.PowerIterationFailedConvergence:
                ppr_cache[u] = {node: 0.0 for node in graph.nodes()}

        # u에서 시작해 v에 도달하는 PageRank 확률 값 추출
        score = ppr_cache[u].get(v, 0.0)
        scores.append(score)

    return scores


# 5. 링크 예측 수행 및 성능 평가
start_time = time.time()

# alpha=0.85 (Damping factor / Restart probability = 0.15)
pos_scores = get_bipartite_pagerank_scores(train_g, test_edges_pos, alpha=0.85)
neg_scores = get_bipartite_pagerank_scores(train_g, test_edges_neg, alpha=0.85)

y_true = [1] * len(pos_scores) + [0] * len(neg_scores)
y_scores = pos_scores + neg_scores

auc = roc_auc_score(y_true, y_scores)
ap = average_precision_score(y_true, y_scores)

# 결과 출력
print(f"-- test pos edge number: {num_test_edges}, test neg edge number: {len(test_edges_neg)}")
print(f"Davis Southern Women 데이터셋 기준 Bipartite PageRank 링크 예측 AUC: {auc:.4f}, AP: {ap:.4f}")
print(f"걸린 시간: {time.time() - start_time:.4f} (s)")

In [1]:
import random
import time
import networkx as nx
from sklearn.metrics import roc_auc_score, average_precision_score

# 1. Davis Southern Women (2-mode) 데이터셋 로드
G = nx.davis_southern_women_graph()

# Node Set U (여성)와 Node Set V (이벤트) 분리
women = [n for n, d in G.nodes(data=True) if d['bipartite'] == 0]
events = [n for n, d in G.nodes(data=True) if d['bipartite'] == 1]

# 전체 엣지 및 테스트 비율 설정
all_edges = list(G.edges())
test_ratio = 0.2  # 테스트 비율 20%
num_test_edges = int(len(all_edges) * test_ratio)

# 2. 테스트용 양성 샘플(Pos) 선택 및 학습용 그래프(Train Graph) 생성
random.seed(42)
test_edges_pos = random.sample(all_edges, num_test_edges)

train_g = G.copy()
train_g.remove_edges_from(test_edges_pos)

# 3. 음성 샘플(Neg) 샘플링 (2-mode 조건 준수: u in U, v in V 간 미연결 쌍만 추출)
test_edges_neg = []
while len(test_edges_neg) < num_test_edges:
    u = random.choice(women)
    v = random.choice(events)
    if not G.has_edge(u, v) and (u, v) not in test_edges_neg:
        test_edges_neg.append((u, v))


# 4. Bipartite Personalised PageRank 계산 함수 정의
def get_bipartite_pagerank_scores(graph, edge_list, alpha=0.85):
    """
    각 시작 노드(u)를 기준(Personalization)으로 PageRank를 수행하여
    목적지 노드(v)에 도달할 확률 점수를 산출
    """
    scores = []
    
    # 시작 노드별로 Personalized PageRank 계산 결과를 캐싱하여 연산 최적화
    ppr_cache = {}

    for u, v in edge_list:
        # 노드 타입 정렬 (u: women, v: events)
        if u in events and v in women:
            u, v = v, u

        # u 노드 기준 Personalized PageRank 계산 (최초 1회만 연산 후 캐싱)
        if u not in ppr_cache:
            # u 노드에서 출발할 확률을 1.0으로 설정하는 personalization 벡터
            personalization = {node: 0.0 for node in graph.nodes()}
            personalization[u] = 1.0
            
            try:
                ppr_cache[u] = nx.pagerank(graph, alpha=alpha, personalization=personalization)
            except nx.PowerIterationFailedConvergence:
                ppr_cache[u] = {node: 0.0 for node in graph.nodes()}

        # u에서 시작해 v에 도달하는 PageRank 확률 값 추출
        score = ppr_cache[u].get(v, 0.0)
        scores.append(score)

    return scores


# 5. 링크 예측 수행 및 성능 평가
start_time = time.time()

# alpha=0.85 (Damping factor / Restart probability = 0.15)
pos_scores = get_bipartite_pagerank_scores(train_g, test_edges_pos, alpha=0.85)
neg_scores = get_bipartite_pagerank_scores(train_g, test_edges_neg, alpha=0.85)

y_true = [1] * len(pos_scores) + [0] * len(neg_scores)
y_scores = pos_scores + neg_scores

auc = roc_auc_score(y_true, y_scores)
ap = average_precision_score(y_true, y_scores)

# 결과 출력
print(f"-- test pos edge number: {num_test_edges}, test neg edge number: {len(test_edges_neg)}")
print(f"Davis Southern Women 데이터셋 기준 Bipartite PageRank 링크 예측 AUC: {auc:.4f}, AP: {ap:.4f}")
print(f"걸린 시간: {time.time() - start_time:.4f} (s)")

-- test pos edge number: 17, test neg edge number: 17
Davis Southern Women 데이터셋 기준 Bipartite PageRank 링크 예측 AUC: 0.6436, AP: 0.7111
걸린 시간: 0.0570 (s)
